[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter2/pgvector-simple.ipynb)

# Chapter 2: Vector Search with pgvector

This notebook demonstrates how to use [pgvector](https://github.com/pgvector/pgvector), the open-source vector similarity search extension for PostgreSQL, to store sentence embeddings and perform nearest-neighbor queries. We embed sentences using Sentence Transformers, insert them into a pgvector-enabled table, and run cosine similarity searches directly in SQL.

**What you'll learn:**
- How to create a pgvector table and HNSW index in PostgreSQL
- How to encode text into embeddings with Sentence Transformers and store them
- How to perform vector similarity search using SQL

**Prerequisites:**
- A running PostgreSQL instance with the `pgvector` extension installed
- `pip install -r requirements.txt`
- Set `PGVECTOR_PASSWORD` environment variable (e.g. `export PGVECTOR_PASSWORD=yourpassword`)

> **Note:** This notebook requires a local PostgreSQL server and will not run out-of-the-box on Google Colab.

### Using PG-Vector for vector search

In this notebook, we will show how to build a vector database in PGVector and commerce a vector search. 

First, we embed a few example sentences using Sentence Transformers. We use the `all-MiniLM-L6-v2` model to turn short sentences like "I am a happy person." into dense numerical vectors. These embeddings will go into the vector database to be looked up later. 

In [2]:
from typing import Final

import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME: Final = "all-MiniLM-L6-v2"
# all-MiniLM-L6-v2 produces 384-dimensional embeddings; the database column below uses the same size.
EMBEDDING_DIMENSION_COUNT: Final = 384
# encode() returns one row per sentence and one column per embedding dimension.
EXPECTED_EMBEDDING_ARRAY_DIMENSION_COUNT: Final = 2


embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

sentences: list[str] = [
    "I am a happy person.",
    "I am a joyful person.",
    "I am a pessimistic person.",
    "I am not an optimistic person.",
]

encoded_sentences: object = embedding_model.encode(sentences, convert_to_numpy=True)

if not isinstance(encoded_sentences, np.ndarray):
    encoded_sentences_type = type(encoded_sentences)
    raise TypeError(
        f"Model {EMBEDDING_MODEL_NAME} must return a NumPy array, got {encoded_sentences_type!r}."
    )

embeddings = encoded_sentences

if embeddings.ndim != EXPECTED_EMBEDDING_ARRAY_DIMENSION_COUNT:
    raise ValueError(
        f"Model {EMBEDDING_MODEL_NAME} returned an array with {embeddings.ndim} dimensions, "
        f"expected {EXPECTED_EMBEDDING_ARRAY_DIMENSION_COUNT}."
    )

actual_dimension_count = embeddings.shape[1]

if actual_dimension_count != EMBEDDING_DIMENSION_COUNT:
    raise ValueError(
        f"Model {EMBEDDING_MODEL_NAME} produced {actual_dimension_count} dimensions, "
        f"expected {EMBEDDING_DIMENSION_COUNT}."
    )

The resulting `embeddings` object is a 2D NumPy array where each row corresponds to a sentence and each column corresponds to one embedding dimension (384 dimensions for `all-MiniLM-L6-v2`). Later, we will store these embeddings in PostgreSQL using the `pgvector` extension so that we can run similarity search directly in the database.

Then, connect to a PostgreSQL instance and create a database called `ragbook`.
We operate on the default database called `postgres` which is the default database in any PG instance. It is the place for performaing administrative tasks. After creating the `ragbook` database, we will switch to `ragbook` for our vector ingestion and serach. 

In [10]:
import os

import psycopg2
import psycopg2.extensions
from psycopg2 import sql
from psycopg2.errors import DuplicateDatabase

POSTGRES_HOST: Final = "localhost"
POSTGRES_PORT: Final = 5432
POSTGRES_USER: Final = "postgres"
# Fail fast instead of waiting on the operating system's TCP timeout when the server is unreachable.
POSTGRES_CONNECT_TIMEOUT_SECONDS: Final = 10
# Every PostgreSQL instance has this database; we use it only for administrative tasks.
ADMIN_DATABASE_NAME: Final = "postgres"
# Lowercase only: a non-default database for our experiments.
VECTOR_DATABASE_NAME: Final = "ragbook"
SENTENCE_EMBEDDINGS_TABLE_NAME: Final = "sentence_embeddings"

PASSWORD_ENVIRONMENT_VARIABLE_NAME: Final = "PGVECTOR_PASSWORD"  # noqa: S105  # variable name, not a secret


class ConfigurationError(Exception):
    """Raised when required notebook configuration is missing."""


def read_required_environment_variable(variable_name: str) -> str:
    """Return the value of an environment variable that must be set and non-empty."""
    variable_value = os.environ.get(variable_name)

    if variable_value is None:
        raise ConfigurationError(f"Environment variable {variable_name} is not set.")

    if variable_value == "":
        raise ConfigurationError(f"Environment variable {variable_name} is empty.")

    return variable_value


def connect_to_database(
    *, database_name: str, password: str
) -> psycopg2.extensions.connection:
    """Open an autocommit connection to one database on the local PostgreSQL server."""
    database_connection = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        user=POSTGRES_USER,
        password=password,
        dbname=database_name,
        connect_timeout=POSTGRES_CONNECT_TIMEOUT_SECONDS,
    )
    # CREATE DATABASE cannot run inside a transaction, so every statement commits immediately.
    database_connection.autocommit = True
    return database_connection


postgres_password = read_required_environment_variable(
    PASSWORD_ENVIRONMENT_VARIABLE_NAME
)

# A database name is an identifier, not a value, so it cannot be a %s parameter.
create_database_statement = sql.SQL("CREATE DATABASE {database_name};").format(
    database_name=sql.Identifier(VECTOR_DATABASE_NAME),
)

admin_connection = connect_to_database(
    database_name=ADMIN_DATABASE_NAME,
    password=postgres_password,
)

# psycopg2's `with connection` ends a transaction but does not close the connection, so close it explicitly.
try:
    admin_cursor = admin_connection.cursor()

    try:
        admin_cursor.execute(create_database_statement)
    except DuplicateDatabase:
        print(f"Database {VECTOR_DATABASE_NAME} already exists.")  # noqa: T201  # notebook output for the reader
    finally:
        # Replaces `with admin_connection.cursor()`, which closed the cursor on exit.
        admin_cursor.close()
finally:
    admin_connection.close()

Database ragbook already exists.


We now switch to the new database. Each cell below opens its own connection to `ragbook` and closes it in a `finally` block, so a cell that fails never leaves a connection open. This cell checks that we can reach `ragbook`.

In [ ]:
ragbook_connection = connect_to_database(
    database_name=VECTOR_DATABASE_NAME,
    password=postgres_password,
)

try:
    with ragbook_connection.cursor() as ragbook_cursor:
        ragbook_cursor.execute("SELECT current_database();")
        current_database_row: object = ragbook_cursor.fetchone()
finally:
    ragbook_connection.close()

if not isinstance(current_database_row, tuple):
    raise TypeError(
        f"Expected one row from current_database(), got {current_database_row!r}."
    )

connected_database_name: object = current_database_row[0]

if connected_database_name != VECTOR_DATABASE_NAME:
    raise ConfigurationError(
        f"Connected to database {connected_database_name!r}, expected {VECTOR_DATABASE_NAME!r}."
    )

print(f"Connected to database {connected_database_name}.")  # noqa: T201  # notebook output for the reader

Connected to database ragbook.


We now create a table called `sentence_embeddings` with two columns: the original sentence text and its embedding. The `embedding` column uses the `VECTOR(384)` type, because the `all-MiniLM-L6-v2` model produces 384-dimensional embeddings.

Before creating the table, we enable the `vector` extension (if it is not already enabled). After creating it, we define an HNSW index on the `embedding` column using `vector_cosine_ops`. This index structure allows PostgreSQL to perform fast approximate nearest-neighbor search over high-dimensional vectors. The operator class must match the distance operator used in queries: `vector_cosine_ops` serves the cosine distance operator `<=>` that our search uses below. An index built with `vector_l2_ops` would only serve `<->` and would be ignored by our query.

In [12]:
# HNSW build parameters; the note below explains the trade-off.
HNSW_MAXIMUM_CONNECTIONS_PER_NODE: Final = 16
HNSW_CONSTRUCTION_CANDIDATE_COUNT: Final = 64

sentence_embeddings_table = sql.Identifier(SENTENCE_EMBEDDINGS_TABLE_NAME)

# Start from an empty table on every run.
drop_table_statement = sql.SQL("DROP TABLE IF EXISTS {table_name};").format(
    table_name=sentence_embeddings_table,
)

create_table_statement = sql.SQL(
    """
    CREATE TABLE {table_name} (
        text TEXT NOT NULL,
        embedding VECTOR({dimension_count})
    );
    """
).format(
    table_name=sentence_embeddings_table,
    dimension_count=sql.Literal(EMBEDDING_DIMENSION_COUNT),
)

# vector_cosine_ops matches the cosine distance operator <=> used by vector_search below.
create_index_statement = sql.SQL(
    """
    CREATE INDEX
    ON {table_name}
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = {maximum_connections_per_node}, ef_construction = {construction_candidate_count});
    """
).format(
    table_name=sentence_embeddings_table,
    maximum_connections_per_node=sql.Literal(HNSW_MAXIMUM_CONNECTIONS_PER_NODE),
    construction_candidate_count=sql.Literal(HNSW_CONSTRUCTION_CANDIDATE_COUNT),
)

ragbook_connection = connect_to_database(
    database_name=VECTOR_DATABASE_NAME,
    password=postgres_password,
)

try:
    with ragbook_connection.cursor() as ragbook_cursor:
        ragbook_cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        ragbook_cursor.execute(drop_table_statement)
        ragbook_cursor.execute(create_table_statement)
        ragbook_cursor.execute(create_index_statement)
finally:
    ragbook_connection.close()

> **Note:** HNSW (Hierarchical Navigable Small World) index parameters control the speed/accuracy trade-off. **m** sets the number of bi-directional links per node — higher values improve recall but use more memory. **ef_construction** controls how many candidates are considered during index building — higher values produce a better-quality graph at the cost of longer build times. The defaults here (m=16, ef_construction=64) work well for small-to-medium datasets.

With the table `sentence_embeddings` ready, we can insert the precomputed embeddings into it, one row at a time, using the SQL `INSERT` statement. Note that the variable `embeddings` we obtained earlier is a 2D NumPy ndarray. Before inserting, we convert each row into a list of Python floats. psycopg2 sends that list as a parameter of the query, and the `::vector` cast in SQL turns it into a pgvector value. We never build SQL by concatenating strings.

In [6]:
from numpy.typing import NDArray


class SentenceInsertError(Exception):
    """Raised when a sentence and its embedding cannot be inserted into the table."""


def convert_embedding_to_float_list(embedding: NDArray[np.float32]) -> list[float]:
    """Convert one embedding row into a list of Python floats for psycopg2."""
    raw_embedding_values: object = embedding.tolist()

    if not isinstance(raw_embedding_values, list):
        raise TypeError(
            f"Embedding must convert to a list, got {raw_embedding_values!r}."
        )

    embedding_values: list[float] = []

    for raw_embedding_value in raw_embedding_values:
        if not isinstance(raw_embedding_value, float):
            raise TypeError(
                f"Embedding values must be floats, got {raw_embedding_value!r}."
            )

        embedding_values.append(raw_embedding_value)

    return embedding_values


insert_sentence_statement = sql.SQL(
    "INSERT INTO {table_name} (text, embedding) VALUES (%s, %s::vector);"
).format(
    table_name=sql.Identifier(SENTENCE_EMBEDDINGS_TABLE_NAME),
)

sentence_count = len(sentences)
embedding_row_count = embeddings.shape[0]

# Check before inserting anything, so a mismatch never leaves a partly filled table.
if sentence_count != embedding_row_count:
    raise ValueError(
        f"Got {sentence_count} sentences but {embedding_row_count} embedding rows."
    )

ragbook_connection = connect_to_database(
    database_name=VECTOR_DATABASE_NAME,
    password=postgres_password,
)

try:
    ragbook_cursor = ragbook_connection.cursor()
    # Autocommit is on, so rows inserted before a failure stay in the table.
    inserted_row_count = 0

    try:
        for sentence_index in range(sentence_count):
            sentence = sentences[sentence_index]
            embedding = embeddings[sentence_index]

            # psycopg2 sends a Python list as a float array, which the ::vector cast converts.
            embedding_values = convert_embedding_to_float_list(embedding)

            ragbook_cursor.execute(
                insert_sentence_statement,
                (sentence, embedding_values),
            )
            inserted_row_count += 1
    except psycopg2.Error as error:
        raise SentenceInsertError(
            f"Inserting sentence {inserted_row_count} failed; "
            f"{inserted_row_count} of {sentence_count} sentences were inserted."
        ) from error
    finally:
        # Replaces `with ragbook_connection.cursor()`, which closed the cursor on exit.
        ragbook_cursor.close()
finally:
    ragbook_connection.close()

Now we can take a sneak peek at the table contents. Each row is a sentence and its embedding. 

In [7]:
select_all_rows_statement = sql.SQL("SELECT text, embedding FROM {table_name};").format(
    table_name=sql.Identifier(SENTENCE_EMBEDDINGS_TABLE_NAME),
)

ragbook_connection = connect_to_database(
    database_name=VECTOR_DATABASE_NAME,
    password=postgres_password,
)

try:
    with ragbook_connection.cursor() as ragbook_cursor:
        ragbook_cursor.execute(select_all_rows_statement)
        stored_rows: object = ragbook_cursor.fetchall()
finally:
    ragbook_connection.close()

if not isinstance(stored_rows, list):
    raise TypeError(f"Expected a list of rows, got {stored_rows!r}.")

for stored_row in stored_rows:
    print(stored_row)  # noqa: T201  # notebook output for the reader

('I am a happy person.', '[0.0046471674,0.0665106,0.014791366,-0.029556928,-0.035561647,-0.027606914,0.12540382,0.04249151,0.012504407,-0.006320998,0.0845007,-0.009377548,0.011230069,-0.02883247,0.059155453,0.017892487,-0.04141187,-0.10653303,0.008443925,0.01986949,-0.08446856,0.0074553792,-0.015888138,0.026268704,-0.026523279,0.020369548,0.049003467,-0.017068269,0.030693017,0.050624005,-0.051756397,-0.009287581,0.111343354,0.0022187566,-0.02579314,0.0023097342,-0.05748578,-0.06991996,0.011813506,0.050560944,0.015797144,-0.011647736,-0.0256201,-0.007323595,0.016757406,-0.025864158,0.07737771,-0.01363742,0.061339527,-0.06329841,-0.003937958,0.0920137,-0.07445012,-0.00904028,0.05114621,0.018042328,0.06534485,-0.002853454,0.0072298977,0.04942297,0.04462305,0.098219015,-0.017557066,0.046022814,0.09118667,-0.006461066,-0.036592428,-0.021893809,-0.052886553,0.00065222196,-0.020817101,0.0062809293,0.09244519,0.014187909,-0.011610371,-0.035127256,0.0822904,0.004944869,0.06444222,0.15541528,0.0

Finally, let's do the vector search. To allow searching again and again, we create a function: 

In [8]:
SEARCH_RESULT_COUNT: Final = 3
# Each search result row holds the sentence text and its similarity score.
SEARCH_RESULT_COLUMN_COUNT: Final = 2


def vector_search(
    query_text: str,
    *,
    sentence_encoder: SentenceTransformer,
    database_cursor: psycopg2.extensions.cursor,
    top_k: int,
) -> list[tuple[str, float]]:
    """Return up to top_k (sentence, cosine similarity) pairs, most similar first."""
    encoded_queries: object = sentence_encoder.encode(
        [query_text], convert_to_numpy=True
    )

    if not isinstance(encoded_queries, np.ndarray):
        encoded_queries_type = type(encoded_queries)
        raise TypeError(
            f"Encoder must return a NumPy array, got {encoded_queries_type!r}."
        )

    query_embedding = encoded_queries[0]
    query_embedding_values = convert_embedding_to_float_list(query_embedding)

    # <=> is cosine distance, so 1 - distance is cosine similarity.
    search_statement = sql.SQL(
        """
        SELECT text,
               1 - (embedding <=> %s::vector) AS similarity
        FROM {table_name}
        WHERE embedding IS NOT NULL
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
        """
    ).format(
        table_name=sql.Identifier(SENTENCE_EMBEDDINGS_TABLE_NAME),
    )

    database_cursor.execute(
        search_statement,
        (query_embedding_values, query_embedding_values, top_k),
    )
    fetched_rows: object = database_cursor.fetchall()

    if not isinstance(fetched_rows, list):
        raise TypeError(f"Expected a list of rows, got {fetched_rows!r}.")

    search_results: list[tuple[str, float]] = []

    for fetched_row in fetched_rows:
        if not isinstance(fetched_row, tuple):
            raise TypeError(f"Each row must be a tuple, got {fetched_row!r}.")

        if len(fetched_row) != SEARCH_RESULT_COLUMN_COUNT:
            raise ValueError(
                f"Each row must have {SEARCH_RESULT_COLUMN_COUNT} columns, got {fetched_row!r}."
            )

        sentence_text: object = fetched_row[0]
        similarity_score: object = fetched_row[1]

        if not isinstance(sentence_text, str):
            raise TypeError(f"Column 'text' must be a string, got {sentence_text!r}.")

        if not isinstance(similarity_score, float):
            raise TypeError(
                f"Column 'similarity' must be a float, got {similarity_score!r}."
            )

        search_results.append((sentence_text, similarity_score))

    return search_results


first_query_text = "I am a smiling person."

ragbook_connection = connect_to_database(
    database_name=VECTOR_DATABASE_NAME,
    password=postgres_password,
)

try:
    with ragbook_connection.cursor() as ragbook_cursor:
        first_search_results = vector_search(
            first_query_text,
            sentence_encoder=embedding_model,
            database_cursor=ragbook_cursor,
            top_k=SEARCH_RESULT_COUNT,
        )
finally:
    ragbook_connection.close()

for first_search_result in first_search_results:
    print(first_search_result)  # noqa: T201  # notebook output for the reader

('I am a happy person.', 0.7639918327331543)
('I am a joyful person.', 0.6934391260147095)
('I am not an optimistic person.', 0.38356049011827986)


The special notation ``vector`` tells PG to treat the field `embedding` as vectors. 
The operation `<=>` performs cosine similarity search. In the SQL `SELECT` statement above, `embedding` refers to the column `embedding` in the table `sentence_embeddings`.

You may wonder why `1 - (embedding <=> %s::vector) as similarity`. This is because in PG Vector, the operation `<=>` in PG-vectgor actually measures the dissimilarity between two vectors. Hence we have to use its complement to measure similarity. 

In the example above, the query sentence is "I am a smiling person." The most similar sentence in the database is "I am a happy person." with a similarity score of 0.76. 

Let's try again. 

In [9]:
second_query_text = "I have a bad feeling about the future."

ragbook_connection = connect_to_database(
    database_name=VECTOR_DATABASE_NAME,
    password=postgres_password,
)

try:
    with ragbook_connection.cursor() as ragbook_cursor:
        second_search_results = vector_search(
            second_query_text,
            sentence_encoder=embedding_model,
            database_cursor=ragbook_cursor,
            top_k=SEARCH_RESULT_COUNT,
        )
finally:
    ragbook_connection.close()

for second_search_result in second_search_results:
    print(second_search_result)  # noqa: T201  # notebook output for the reader

('I am a pessimistic person.', 0.5613031910296491)
('I am not an optimistic person.', 0.5059884341355694)
('I am a happy person.', 0.32247012853622437)


This time, the query is "I have a bad feeling about the future." The most similar sentence in the database is "I am a pessimistic person." with a similarity score of 0.56, followed by "I am not an optimistic person." at 0.51. It totally makes sense.